# MANTA Model Inspection
This notebook inspects model internals: parameter counts, symmetric kernels, EllipticMish shape, and attention responses.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

from manta.models.manta import MANTA
from manta.models.components.elliptic_activation import EllipticMish
from manta.utils.config import MANTAConfig

## Step 1: Instantiate model and inspect parameter counts
We build the default MANTA config and print per-component parameter budgets.

In [ ]:
config = MANTAConfig()
model = MANTA(config)
param_counts = model.get_parameter_count()
param_counts

## Step 2: Visualize palindromic kernels
PalindromicConv1d weights should be symmetric by construction to encode ingress/egress reversibility.

In [ ]:
kernels = model.local_encoder.get_symmetric_kernels()
fig, axs = plt.subplots(len(kernels), 1, figsize=(8, 2.2 * len(kernels)))
if len(kernels) == 1:
    axs = [axs]
for i, kernel in enumerate(kernels):
    k = kernel[0, 0].numpy()
    axs[i].plot(k, marker='o')
    axs[i].plot(k[::-1], linestyle='--', alpha=0.7)
    axs[i].set_title(f'Palindromic Kernel Layer {i + 1}')
    axs[i].grid(alpha=0.2)
plt.tight_layout()
plt.show()

## Step 3: Plot EllipticMish activation curve
Compare smooth non-monotonic response to standard nonlinearities.

In [ ]:
elliptic = EllipticMish(alpha_init=0.1)
x, y = elliptic.get_activation_curve()

x_t = torch.tensor(x, dtype=torch.float32)
relu = torch.relu(x_t).numpy()
mish = (x_t * torch.tanh(torch.nn.functional.softplus(x_t))).numpy()

plt.figure(figsize=(8, 4.5))
plt.plot(x, y, label='EllipticMish', lw=2.0)
plt.plot(x, relu, label='ReLU', lw=1.5)
plt.plot(x, mish, label='Mish', lw=1.5)
plt.legend(frameon=False)
plt.title('Activation Comparison')
plt.xlabel('x')
plt.ylabel('f(x)')
plt.grid(alpha=0.2)
plt.show()

## Step 4: Forward pass and attention extraction
We run one synthetic batch through MANTA and inspect global/local attention maps.

In [ ]:
batch = {
    'global_view': torch.randn(2, 1, config.model.global_input_length),
    'local_view': torch.randn(2, 1, config.model.local_input_length),
    'freq_bands': torch.randn(2, 3, config.model.global_input_length),
    'stellar_params': torch.tensor([[5777.0, 4.4, 0.0, 22.0], [5200.0, 4.3, -0.2, 16.0]], dtype=torch.float32),
    'label': torch.tensor([1.0, 0.0], dtype=torch.float32),
}

with torch.no_grad():
    pred = model(batch)

print('Predictions:', pred.squeeze().tolist())
g_attn = model.global_attention.get_last_attention()
l_attn = model.local_attention.get_last_attention()

if g_attn is not None:
    plt.figure(figsize=(5, 4))
    plt.imshow(g_attn[0, 0].cpu().numpy(), aspect='auto', cmap='inferno')
    plt.title('Global Attention (Head 1)')
    plt.colorbar()
    plt.show()

if l_attn is not None:
    plt.figure(figsize=(5, 4))
    plt.imshow(l_attn[0, 0].cpu().numpy(), aspect='auto', cmap='inferno')
    plt.title('Local Attention (Head 1)')
    plt.colorbar()
    plt.show()